# 02.08 - Segmentation fundamentals + U-Net

**Notebook type:** Practice notebook with theory, exercises, TODO cells, and test cases.

**Daily output:** Validated segmentation Dataset + compact U-Net.

You will build an image-mask data boundary, apply synchronized geometry, implement a compact U-Net, and verify multiclass output with Dice and IoU.

## Core Ideas

- **Semantic segmentation** assigns a class to every pixel; **instance segmentation** also separates individual objects of the same class.
- Multiclass masks store integer class IDs in shape `[H, W]`. `CrossEntropyLoss` expects logits `[N, C, H, W]` and `int64` targets `[N, H, W]`; do not one-hot encode those targets.
- Image transforms may use bilinear interpolation, but masks require nearest-neighbor interpolation so class IDs are not blended. Random geometric decisions must be shared by image and mask.
- U-Net downsamples to learn context, upsamples to recover resolution, and concatenates encoder features through skip connections to restore spatial detail.
- Dice and IoU are overlap metrics. Report per-class values because background can dominate the mean.

## Allowed-Library Pretrained Route

Torchvision does not provide U-Net, so implementing compact U-Net is justified because its encoder-decoder and skip connections are the explicit lesson objective. For a competition baseline, the allowlist does provide official pretrained semantic-segmentation architectures: `deeplabv3_resnet50`, `deeplabv3_mobilenet_v3_large`, `fcn_resnet50`, and `lraspp_mobilenet_v3_large`. Use their `..._Weights.DEFAULT` checkpoints only when permitted and cached/attached; use `weights=None, weights_backbone=None` for architecture-only offline construction.

Official reference: https://docs.pytorch.org/vision/stable/models/deeplabv3.html

In [ ]:
import os
import csv
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

SEED = 2
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_ROOT = "_day02_segmentation_data"
IMAGE_DIR = os.path.join(DATA_ROOT, "images")
MASK_DIR = os.path.join(DATA_ROOT, "masks")
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(MASK_DIR, exist_ok=True)
CLASS_NAMES = ["background", "circle", "rectangle"]
import pandas as pd

from sklearn.model_selection import StratifiedKFold


## Prepared Image and Mask Data

The provided cell writes 18 RGB images, 18 single-channel PNG masks, and a manifest. Masks use exact IDs `{0, 1, 2}`. File-backed fixtures make it possible to test path alignment, mode conversion, and label preservation.

In [ ]:
manifest_rows = []
for index in range(18):
    image = Image.new("RGB", (64, 64), color=(225, 228, 232))
    mask = Image.new("L", (64, 64), color=0)
    image_draw = ImageDraw.Draw(image)
    mask_draw = ImageDraw.Draw(mask)
    cx = 16 + (index * 5) % 25
    cy = 17 + (index * 7) % 23
    radius = 7 + index % 4
    circle_box = (cx - radius, cy - radius, cx + radius, cy + radius)
    image_draw.ellipse(circle_box, fill=(220, 75, 70))
    mask_draw.ellipse(circle_box, fill=1)
    x1 = 35 + index % 7
    y1 = 38 - index % 6
    rectangle_box = (x1, y1, x1 + 18, y1 + 14)
    image_draw.rectangle(rectangle_box, fill=(55, 130, 225))
    mask_draw.rectangle(rectangle_box, fill=2)
    image_name = f"image_{index:02d}.png"
    mask_name = f"mask_{index:02d}.png"
    image.save(os.path.join(IMAGE_DIR, image_name))
    mask.save(os.path.join(MASK_DIR, mask_name))
    manifest_rows.append({"image_path": os.path.join("images", image_name), "mask_path": os.path.join("masks", mask_name)})

with open(os.path.join(DATA_ROOT, "labels.csv"), "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=["image_path", "mask_path"])
    writer.writeheader()
    writer.writerows(manifest_rows)

print("prepared pairs:", len(manifest_rows))
print("mask IDs:", np.unique(np.asarray(Image.open(os.path.join(MASK_DIR, "mask_00.png")))).tolist())

## Exercise 02-A: Build an image-mask Dataset

Load paths from the manifest rows. Convert RGB pixels to `float32 [0,1]` in channel-first layout, and masks to unscaled class IDs. Assert that spatial shapes match and mask IDs are valid.

**Return structure — `SegmentationDataset`:** A `torch.utils.data.Dataset` of length `N`. `dataset[i]` returns a dictionary with `image`: CPU `torch.float32 [3,H,W]`; `mask`: CPU `torch.int64 [H,W]`; and `image_path`, `mask_path`: Python strings. Pixel values are in `[0,1]`, and mask values belong to `[0, num_classes-1]`.

In [ ]:
# TODO 02-A
class SegmentationDataset(Dataset):
    def __init__(self, rows, root=DATA_ROOT, num_classes=3):
        raise NotImplementedError("Complete Exercise 02-A")

    def __len__(self):
        raise NotImplementedError("Complete Exercise 02-A")

    def __getitem__(self, index):
        raise NotImplementedError("Complete Exercise 02-A")


# Smoke check: run this after implementing the class above.
segmentation_dataset = SegmentationDataset(manifest_rows)
segmentation_sample = segmentation_dataset[0]
print(segmentation_sample["image"].shape, segmentation_sample["image"].dtype)
print(segmentation_sample["mask"].shape, segmentation_sample["mask"].dtype, torch.unique(segmentation_sample["mask"]).tolist())

## Exercise 02-B: Apply synchronized geometric augmentation

Use `torchvision.transforms.functional.hflip`. The same Boolean decision must control both image and mask. A horizontal flip preserves dtype and exact mask IDs.

**Return structure — `synchronized_horizontal_flip`:** A tuple `(image_out, mask_out)`. Position 0 is a tensor with the same shape, dtype, and device as input image `[C,H,W]`. Position 1 is a tensor with the same shape, dtype, and device as input mask `[H,W]`. Inputs are not modified.

In [ ]:
# TODO 02-B
def synchronized_horizontal_flip(image, mask, apply_flip):
    raise NotImplementedError("Complete Exercise 02-B")


# Smoke check: run this after implementing the function above.
flipped_image, flipped_mask = synchronized_horizontal_flip(segmentation_sample["image"], segmentation_sample["mask"], True)
print("flip preserved mask IDs:", torch.unique(flipped_mask).tolist())
print("alignment check:", bool(torch.equal(flipped_image[:, :, 0], segmentation_sample["image"][:, :, -1])))

## Exercise 02-C: Implement a compact U-Net

Build two encoder stages, a bottleneck, two upsampling stages, skip concatenations, and a `1 x 1` output convolution. If a transposed convolution and skip tensor differ by a pixel, resize the decoder tensor with bilinear interpolation before concatenation.

**Return structure — `DoubleConv`:** A callable `nn.Module`. Calling with floating tensor `[N,C_in,H,W]` returns floating tensor `[N,C_out,H,W]` on the same device.

**Return structure — `CompactUNet`:** A callable `nn.Module`. Construction returns a module with `num_classes` output channels. Calling with `torch.float32 [N,3,H,W]`, where `H` and `W` are divisible by four, returns raw logits `torch.float32 [N,num_classes,H,W]` on the input device.

In [ ]:
# TODO 02-C
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        raise NotImplementedError("Complete Exercise 02-C")

    def forward(self, inputs):
        raise NotImplementedError("Complete Exercise 02-C")


class CompactUNet(nn.Module):
    def __init__(self, num_classes=3, base_channels=16):
        super().__init__()
        raise NotImplementedError("Complete Exercise 02-C")

    def forward(self, images):
        raise NotImplementedError("Complete Exercise 02-C")


# Smoke check: run this after implementing both classes above.
unet = CompactUNet(num_classes=3)
segmentation_logits = unet(segmentation_sample["image"].unsqueeze(0))
print("U-Net logits:", segmentation_logits.shape, segmentation_logits.dtype, segmentation_logits.device)

## Exercise 02-D: Calculate per-class Dice and IoU

Accept predicted class IDs and target class IDs. For a class absent from both prediction and target, return `NaN` rather than a misleading perfect score. Average only finite class results.

**Return structure — `segmentation_metrics`:** A dictionary with `dice` and `iou`, Python lists of length `num_classes` containing floats or `float('nan')`; `mean_dice` and `mean_iou`, Python floats averaged over finite classes; and `support`, a Python list of `num_classes` integer target-pixel counts.

In [ ]:
# TODO 02-D
def segmentation_metrics(predicted_mask, target_mask, num_classes=3):
    raise NotImplementedError("Complete Exercise 02-D")


# Smoke check: run this after implementing the function above.
perfect_metrics = segmentation_metrics(segmentation_sample["mask"], segmentation_sample["mask"], 3)
print("perfect-mask metrics:", perfect_metrics)

## Exercise 02-E: Verify the loss boundary and visualize predictions

Run one batch through U-Net, calculate multiclass cross-entropy directly from logits, choose pixel classes with `argmax`, and compute metrics. Do not apply softmax before `CrossEntropyLoss`.

**Return structure — `inspect_segmentation_batch`:** A dictionary with `logits`: floating tensor `[N,C,H,W]`; `predicted_masks`: `torch.int64 [N,H,W]`; `loss`: scalar differentiable tensor; and `metrics`: a list of `N` dictionaries following the exact `segmentation_metrics` schema. Tensor outputs remain on the model/input device.

In [ ]:
# TODO 02-E
def inspect_segmentation_batch(model, images, masks):
    raise NotImplementedError("Complete Exercise 02-E")


# Smoke check: run this after implementing the function above.
demo_loader = DataLoader(segmentation_dataset, batch_size=3, shuffle=False)
demo_batch = next(iter(demo_loader))
batch_report = inspect_segmentation_batch(unet, demo_batch["image"], demo_batch["mask"])
print("batch loss:", float(batch_report["loss"].detach()))
print("first-image metrics:", batch_report["metrics"][0])

plt.figure(figsize=(9, 3))
plt.subplot(1, 3, 1); plt.imshow(demo_batch["image"][0].permute(1, 2, 0)); plt.title("Image"); plt.axis("off")
plt.subplot(1, 3, 2); plt.imshow(demo_batch["mask"][0], vmin=0, vmax=2); plt.title("True mask"); plt.axis("off")
plt.subplot(1, 3, 3); plt.imshow(batch_report["predicted_masks"][0].detach(), vmin=0, vmax=2); plt.title("Untrained prediction"); plt.axis("off")
plt.tight_layout(); plt.show()

## Exercise 02-F: Prepare stratified segmentation folds

All prepared masks contain the same three classes, so class presence cannot create useful strata. Instead, rank images by foreground-pixel fraction and divide them into low, medium, and high coverage strata of equal size, then apply `StratifiedKFold`. This is a documented proxy for the toy data. Real segmentation competitions may require multilabel class-presence vectors, pixel-frequency bins, and group-aware constraints together.

**Return structure — `make_segmentation_folds`:** A dictionary with `folds`, a list of exactly `n_splits` dictionaries containing `fold`, `train_indices`, `val_indices`, `train_stratum_support`, and `val_stratum_support`; and `assignments`, a `pandas.DataFrame` of length `N` with columns `row_index`, `image_path`, `foreground_fraction`, `coverage_stratum`, and `fold`. Every row index appears in exactly one validation list. The function also writes the assignments to `_day02_segmentation_data/fold_assignments.csv`.

In [ ]:
# TODO 02-F
def make_segmentation_folds(rows, root=DATA_ROOT, n_splits=3, seed=SEED):
    raise NotImplementedError("Complete Exercise 02-F")


# Smoke check: inspect coverage-stratum support for every fold.
segmentation_fold_plan = make_segmentation_folds(manifest_rows)
print(pd.DataFrame([{key: fold[key] for key in ["fold", "train_stratum_support", "val_stratum_support"]} for fold in segmentation_fold_plan["folds"]]).to_string(index=False))

## Test Cases

These tests validate file alignment, mask integrity, synchronized transforms, U-Net shape, loss compatibility, and metric accounting.

**Return structure — `run_day02_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 02 tests passed`.

In [ ]:
def run_day02_tests():
    assert os.path.isfile(os.path.join(DATA_ROOT, "labels.csv"))
    assert len(os.listdir(IMAGE_DIR)) == 18 and len(os.listdir(MASK_DIR)) == 18
    assert len(segmentation_dataset) == 18
    item = segmentation_dataset[0]
    assert set(item) == {"image", "mask", "image_path", "mask_path"}
    assert item["image"].shape == (3, 64, 64) and item["image"].dtype == torch.float32
    assert item["mask"].shape == (64, 64) and item["mask"].dtype == torch.int64
    assert set(torch.unique(item["mask"]).tolist()) == {0, 1, 2}
    no_flip_image, no_flip_mask = synchronized_horizontal_flip(item["image"], item["mask"], False)
    yes_flip_image, yes_flip_mask = synchronized_horizontal_flip(item["image"], item["mask"], True)
    assert torch.equal(no_flip_image, item["image"]) and torch.equal(no_flip_mask, item["mask"])
    assert torch.equal(yes_flip_image, torch.flip(item["image"], dims=[2]))
    assert torch.equal(yes_flip_mask, torch.flip(item["mask"], dims=[1]))
    with torch.no_grad():
        logits = unet(torch.stack([item["image"], item["image"]]))
    assert logits.shape == (2, 3, 64, 64) and logits.dtype == torch.float32
    metrics = segmentation_metrics(item["mask"], item["mask"], 3)
    assert metrics["support"] == torch.bincount(item["mask"].flatten(), minlength=3).tolist()
    assert np.allclose(metrics["dice"], [1.0, 1.0, 1.0])
    assert np.allclose(metrics["iou"], [1.0, 1.0, 1.0])
    assert batch_report["logits"].shape == (3, 3, 64, 64)
    assert batch_report["predicted_masks"].dtype == torch.int64
    assert len(batch_report["metrics"]) == 3
    fold_validation = [index for fold in segmentation_fold_plan["folds"] for index in fold["val_indices"]]
    assert sorted(fold_validation) == list(range(len(manifest_rows)))
    assert all(fold["train_stratum_support"] == [4, 4, 4] for fold in segmentation_fold_plan["folds"])
    assert all(fold["val_stratum_support"] == [2, 2, 2] for fold in segmentation_fold_plan["folds"])
    assert segmentation_fold_plan["assignments"]["fold"].tolist().count(-1) == 0
    assert os.path.isfile(os.path.join(DATA_ROOT, "fold_assignments.csv"))
    print("Day 02 tests passed")


run_day02_tests()

## Day 02 Checklist

- [ ] I can distinguish semantic from instance segmentation.
- [ ] My masks remain `int64 [H,W]` with exact class IDs.
- [ ] I synchronize geometric changes and use nearest-neighbor logic for masks.
- [ ] I can trace U-Net encoder, bottleneck, decoder, and skip shapes.
- [ ] My model returns raw `[N,C,H,W]` logits compatible with `CrossEntropyLoss`.
- [ ] I report per-class Dice/IoU and inspect qualitative masks.
- [ ] I prepared segmentation folds using a documented mask-coverage stratification proxy.
